# 04 — Call Summarization

**Goal:** for every processed call, a short summary of the **customer issue**, the **agent action**, and the **outcome or next step**.

**Runs on:** Google Colab with a GPU (Runtime → Change runtime type → T4 GPU).

**Input:** `results/segmented_turns.csv` — our own Whisper transcripts with speaker roles (notebook 03). The human reference transcripts are **not** used here.

**Output**
- `results/summaries.csv` — one summary per call, with automatic check results
- `results/summary_review.csv` — 10 calls prepared for manual review
- `results/summarization_run_info.json` — model and library versions

**Sections**
1. Choosing a summarization approach
2. Setup
3. Load the language model
4. The prompt
5. Generate the summaries
6. Automatic checks
7. Manual review sheet
8. Findings
9. Save and download (Colab)

## 1. Choosing a summarization approach

| Approach | How it works | Fit for this task |
|---|---|---|
| **Extractive** | Picks existing sentences from the call | Poor: no single sentence states issue + action + outcome; spoken sentences are fragmented |
| **Rule-based** | Templates and keywords (for example "transfer" → issue = transfer) | Works only for the 8 known intents; cannot describe what actually happened |
| **Abstractive (model-based)** | A language model writes new sentences | Good fit: flexible and readable. **Risk:** it can invent details (hallucination) |

We use **abstractive summarization with a small open-source instruction-tuned model** (`Qwen3-4B-Instruct-2507`, Apache-2.0 license; fallback `Qwen2.5-1.5B-Instruct`). It runs on a free Colab GPU, needs no paid API, and no data leaves the notebook.

**How we limit hallucination**
1. The prompt says: use only what is stated; write *"not stated"* otherwise.
2. **Greedy decoding** (no random sampling): the same transcript always gives the same summary.
3. A fixed **JSON format** with three fields, so every summary has the same structure.
4. **Automatic checks** (section 6) and a **manual review** of 10 calls (section 7).

## 2. Setup

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ahmedalmoutiry-wq/call-center-agent-performance-analysis.git"
REPO_DIR = "/content/call-center-agent-performance-analysis"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    os.chdir(REPO_DIR)
    !pip install -q num2words==0.5.14

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import RESULTS_DIR
from src.llm import DEFAULT_MODELS, load_llm
from src.summarize import (NOT_STATED, SUMMARY_FIELDS, build_messages, issue_matches_task,
                           parse_summary, transcript_text, ungrounded_numbers)

SEED = 42
pd.set_option("display.max_colwidth", 120)

turns = pd.read_csv(RESULTS_DIR / "segmented_turns.csv")
subset = pd.read_csv(RESULTS_DIR / "subset_ids.csv")
print(f"Calls: {turns['call_id'].nunique()}  |  turns: {len(turns)}")

## 3. Load the language model

In [ ]:
generate, MODEL_ID = load_llm(DEFAULT_MODELS)

import torch, transformers
run_info = {"model": MODEL_ID, "transformers": transformers.__version__, "torch": torch.__version__,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
            "decoding": "greedy (do_sample=False)", "max_new_tokens": 200}
run_info

## 4. The prompt

The **system message** sets the rules (only stated facts, "not stated" otherwise). The **user message** contains the transcript and the output format. Here is the full prompt for one call:

In [ ]:
example_call = turns["call_id"].iloc[0]
for message in build_messages(turns[turns["call_id"] == example_call]):
    print(f"--- {message['role'].upper()} ---")
    print(message["content"])
    print()

## 5. Generate the summaries

In [ ]:
rows, started = [], time.perf_counter()
for call_id, call_turns in turns.groupby("call_id"):
    raw = generate(build_messages(call_turns))
    rows.append({"call_id": call_id, **parse_summary(raw), "raw_output": raw})

summaries = pd.DataFrame(rows)
print(f"Generated {len(summaries)} summaries in {time.perf_counter() - started:.0f} s")
summaries[["call_id"] + SUMMARY_FIELDS].head(5)

## 6. Automatic checks

These checks need no human labels. They cannot prove a summary is good, but they catch clear problems:

| Check | What it catches |
|---|---|
| **Format** — all three fields present | The model did not follow the JSON format |
| **Issue matches the known task** — the issue mentions the call's task type (for example "transfer") | A wrong customer issue. The task type comes from the dataset metadata, which we did not give the model |
| **Numbers are grounded** — every amount, time or number in the summary appears in the transcript | Invented details (hallucination) |
| **"not stated" count** | How often the model admits information is missing |

In [ ]:
summaries = summaries.merge(subset[["call_id", "task_type"]], on="call_id")
transcripts = {c: transcript_text(t) for c, t in turns.groupby("call_id")}

summaries["issue_matches_task"] = [issue_matches_task(i, t)
                                   for i, t in zip(summaries["customer_issue"], summaries["task_type"])]
summaries["ungrounded_numbers"] = [
    ", ".join(ungrounded_numbers(" ".join(r[f] for f in SUMMARY_FIELDS), transcripts[r["call_id"]]))
    for _, r in summaries.iterrows()]

checks = pd.Series({
    "summaries": len(summaries),
    "format OK": int(summaries["parse_ok"].sum()),
    "issue matches the known task": int(summaries["issue_matches_task"].sum()),
    "summaries with ungrounded numbers": int((summaries["ungrounded_numbers"] != "").sum()),
    **{f'"{NOT_STATED}" in {f}': int(summaries[f].str.lower().str.contains(NOT_STATED).sum()) for f in SUMMARY_FIELDS},
})
checks.to_frame("count")

In [ ]:
# Calls that failed a check, to read them closely
problems = summaries[~summaries["parse_ok"] | ~summaries["issue_matches_task"] | (summaries["ungrounded_numbers"] != "")]
problems[["call_id", "task_type", "customer_issue", "outcome", "ungrounded_numbers"]]

In [ ]:
# Read one flagged call next to its transcript (if any call was flagged)
if len(problems):
    call_id = problems["call_id"].iloc[0]
    print(transcripts[call_id], "\n")
    print(summaries.loc[summaries["call_id"] == call_id, SUMMARY_FIELDS].T.to_string())

## 7. Manual review sheet

Automatic checks cannot judge whether a summary is **faithful** (says nothing false) and **complete** (covers issue, action and outcome). A person must read them.

We pick 10 calls at random (fixed seed), always including the edge-case call. The sheet `results/summary_review.csv` has empty columns to fill in after reading each transcript and summary:
- `faithful` — `yes` / `no`: is every statement supported by the transcript?
- `complete` — `yes` / `no`: are issue, action and outcome all correct (or correctly "not stated")?
- `review_notes` — what is wrong, if anything

In [ ]:
edge_call = subset.loc[subset["agent_on_channel_1"], "call_id"].tolist()
others = summaries.loc[~summaries["call_id"].isin(edge_call), "call_id"].sample(n=10 - len(edge_call), random_state=SEED)
review_ids = edge_call + others.tolist()

review = summaries[summaries["call_id"].isin(review_ids)][["call_id", "task_type"] + SUMMARY_FIELDS].copy()
review.insert(2, "transcript", review["call_id"].map(transcripts))
review["faithful"], review["complete"], review["review_notes"] = "", "", ""
review.to_csv(RESULTS_DIR / "summary_review.csv", index=False)
print(f"Review sheet: {len(review)} calls -> results/summary_review.csv")

In [ ]:
# Print the 10 calls to read them here
for r in review.itertuples():
    print(f"===== {r.call_id}  ({r.task_type}) =====")
    print(r.transcript)
    print(f"\n  ISSUE:   {r.customer_issue}\n  ACTION:  {r.agent_action}\n  OUTCOME: {r.outcome}\n")

## 8. Findings

_To be written after the run, from the numbers above and the manual review._

## 9. Save and download (Colab)

In [ ]:
out_cols = ["call_id", "task_type"] + SUMMARY_FIELDS + ["parse_ok", "issue_matches_task", "ungrounded_numbers", "raw_output"]
summaries.assign(model=MODEL_ID)[out_cols + ["model"]].to_csv(RESULTS_DIR / "summaries.csv", index=False)
(RESULTS_DIR / "summarization_run_info.json").write_text(json.dumps(run_info, indent=2))

if IN_COLAB:
    import shutil
    from google.colab import files
    export_dir = Path("/content/summary_results")
    export_dir.mkdir(exist_ok=True)
    for name in ["summaries.csv", "summary_review.csv", "summarization_run_info.json"]:
        shutil.copy(RESULTS_DIR / name, export_dir / name)
    shutil.make_archive("/content/summary_results", "zip", export_dir)
    files.download("/content/summary_results.zip")